#  <center>Taller de Aprendizaje Automatico</center>
##  <center>Paso 3: Procesamiento de datos</center>
### <center>Procesamiento base, intento 1</center>

Este notebook toma el flujo base de procesamiento y le agrega una primera decision concreta de preprocesamiento: eliminar las features cuyo marcador `-999` supera el `70%` en el conjunto de entrenamiento.

La idea es que este intento quede como una variante simple y trazable. Primero se cargan los datos, despues se separan identificadores, target y features, y por ultimo se aplica el descarte de columnas sobre `X_train`, replicando exactamente la misma decision en `X_test`.

## Carga reproducible del dataset

Se cargan `training.csv` y `test.csv` desde `data/higgs-boson/`. El archivo de entrenamiento contiene `Label` y `Weight`; el archivo de test no.

In [1]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from IPython.display import display


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "roadmap.md").exists() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("No se pudo localizar la raiz del proyecto.")


project_root = find_project_root()
data_dir = project_root / "data" / "higgs-boson"
train_path = data_dir / "training.csv"
test_path = data_dir / "test.csv"

if not train_path.exists():
    raise FileNotFoundError(f"No se encontro el archivo esperado: {train_path}")

if not test_path.exists():
    raise FileNotFoundError(f"No se encontro el archivo esperado: {test_path}")

train_data = pd.read_csv(train_path)
test_data = pd.read_csv(test_path)

print(f"project_root = {project_root}")
print(f"train shape = {train_data.shape}")
print(f"test shape = {test_data.shape}")


project_root = /home/santig14/fing/taa/1_TallerAA
train shape = (250000, 33)
test shape = (550000, 31)


## Verificacion visual de la carga

Se muestran las primeras filas de training y test para dejar evidencia de que la carga fue correcta.

In [2]:
display(train_data.head())
display(test_data.head())


,EventId,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,...,PRI_jet_num,PRI_jet_leading_pt,PRI_jet_leading_eta,PRI_jet_leading_phi,PRI_jet_subleading_pt,PRI_jet_subleading_eta,PRI_jet_subleading_phi,PRI_jet_all_pt,Weight,Label
0,100000,138.470,51.655,97.827,27.980,0.91,124.711,2.666,3.064,41.928,...,2,67.435,2.150,0.444,46.062,1.24,-2.475,113.497,0.002653,s
1,100001,160.937,68.768,103.235,48.146,-999.00,-999.000,-999.000,3.473,2.078,...,1,46.226,0.725,1.158,-999.000,-999.00,-999.000,46.226,2.233584,b
2,100002,-999.000,162.172,125.953,35.635,-999.00,-999.000,-999.000,3.148,9.336,...,1,44.251,2.053,-2.028,-999.000,-999.00,-999.000,44.251,2.347389,b
3,100003,143.905,81.417,80.943,0.414,-999.00,-999.000,-999.000,3.310,0.414,...,0,-999.000,-999.000,-999.000,-999.000,-999.00,-999.000,-0.000,5.446378,b
4,100004,175.864,16.915,134.805,16.405,-999.00,-999.000,-999.000,3.891,16.405,...,0,-999.000,-999.000,-999.000,-999.000,-999.00,-999.000,0.000,6.245333,b


,EventId,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,...,PRI_met_phi,PRI_met_sumet,PRI_jet_num,PRI_jet_leading_pt,PRI_jet_leading_eta,PRI_jet_leading_phi,PRI_jet_subleading_pt,PRI_jet_subleading_eta,PRI_jet_subleading_phi,PRI_jet_all_pt
0,350000,-999.000,79.589,23.916,3.036,-999.000,-999.000,-999.000,0.903,3.036,...,2.022,98.556,0,-999.000,-999.000,-999.000,-999.000,-999.000,-999.000,-0.000
1,350001,106.398,67.490,87.949,49.994,-999.000,-999.000,-999.000,2.048,2.679,...,-1.138,176.251,1,47.575,-0.553,-0.849,-999.000,-999.000,-999.000,47.575
2,350002,117.794,56.226,96.358,4.137,-999.000,-999.000,-999.000,2.755,4.137,...,-1.868,111.505,0,-999.000,-999.000,-999.000,-999.000,-999.000,-999.000,0.000
3,350003,135.861,30.604,97.288,9.104,-999.000,-999.000,-999.000,2.811,9.104,...,1.172,164.707,0,-999.000,-999.000,-999.000,-999.000,-999.000,-999.000,0.000
4,350004,74.159,82.772,58.731,89.646,1.347,536.663,-0.339,1.028,77.213,...,-0.231,869.614,3,254.085,-1.013,-0.334,185.857,0.335,2.587,599.213


## Separacion base para entrenamiento

Antes de cualquier transformacion, se separan las columnas que no deben entrar como features del modelo:

- `EventId`: identificador del evento.
- `Weight`: peso asociado al evento, que no se usa como feature.
- `Label`: variable objetivo, que pasa a `y_train`.

Con eso se arman `X_train`, `y_train` y `X_test` como punto de partida comun.

In [3]:
excluded_train_columns = ["EventId", "Weight", "Label"]
feature_columns = [
    column for column in train_data.columns
    if column not in excluded_train_columns
]

train_event_id = train_data["EventId"].copy()
train_weight = train_data["Weight"].copy()
y_train = train_data["Label"].copy()
X_train = train_data[feature_columns].copy()

test_event_id = test_data["EventId"].copy()
X_test = test_data[feature_columns].copy()

expected_test_columns = [column for column in test_data.columns if column != "EventId"]
if feature_columns != expected_test_columns:
    raise ValueError("Las columnas de entrenamiento y test no coinciden despues de excluir EventId.")

print(f"Cantidad de features iniciales: {len(feature_columns)}")
print(f"X_train shape = {X_train.shape}")
print(f"y_train shape = {y_train.shape}")
print(f"X_test shape = {X_test.shape}")


Cantidad de features iniciales: 30
X_train shape = (250000, 30)
y_train shape = (250000,)
X_test shape = (550000, 30)


## Regla del intento 1

En este intento se usa un criterio simple: si una feature tiene mas de `70%` de valores `-999` en training, entonces se elimina del conjunto de entrada. La decision se toma usando solo `X_train`, y despues se replica exactamente sobre `X_test`.

In [4]:
missing_marker = -999
missing_threshold_pct = 70.0

marker_rows = []
for column in X_train.columns:
    train_marker_count = int((X_train[column] == missing_marker).sum())
    train_marker_pct = round(train_marker_count / len(X_train) * 100, 2)
    marker_rows.append(
        {
            "feature": column,
            "train_minus_999_count": train_marker_count,
            "train_minus_999_pct": train_marker_pct,
        }
    )

marker_summary = pd.DataFrame(marker_rows).sort_values(
    ["train_minus_999_pct", "feature"],
    ascending=[False, True],
)

drop_columns_i1 = marker_summary.loc[
    marker_summary["train_minus_999_pct"] > missing_threshold_pct,
    "feature",
].tolist()

display(marker_summary.loc[marker_summary["feature"].isin(drop_columns_i1)].reset_index(drop=True))
print(f"Columnas a eliminar en intento 1: {len(drop_columns_i1)}")
print(drop_columns_i1)


,feature,train_minus_999_count,train_minus_999_pct
0,DER_deltaeta_jet_jet,177457,70.98
1,DER_lep_eta_centrality,177457,70.98
2,DER_mass_jet_jet,177457,70.98
3,DER_prodeta_jet_jet,177457,70.98
4,PRI_jet_subleading_eta,177457,70.98
5,PRI_jet_subleading_phi,177457,70.98
6,PRI_jet_subleading_pt,177457,70.98


Columnas a eliminar en intento 1: 7
['DER_deltaeta_jet_jet', 'DER_lep_eta_centrality', 'DER_mass_jet_jet', 'DER_prodeta_jet_jet', 'PRI_jet_subleading_eta', 'PRI_jet_subleading_phi', 'PRI_jet_subleading_pt']


## Aplicacion del descarte

Las columnas detectadas se eliminan tanto de `X_train` como de `X_test`. A partir de este punto, estas dos nuevas tablas son la base del intento 1.

In [5]:
X_train_i1 = X_train.drop(columns=drop_columns_i1).copy()
X_test_i1 = X_test.drop(columns=drop_columns_i1).copy()
feature_columns_i1 = X_train_i1.columns.tolist()

print(f"X_train_i1 shape = {X_train_i1.shape}")
print(f"X_test_i1 shape = {X_test_i1.shape}")
print(f"Cantidad de features despues del descarte: {len(feature_columns_i1)}")

display(pd.DataFrame({"dropped_feature": drop_columns_i1}))


X_train_i1 shape = (250000, 23)
X_test_i1 shape = (550000, 23)
Cantidad de features despues del descarte: 23


,dropped_feature
0,DER_deltaeta_jet_jet
1,DER_lep_eta_centrality
2,DER_mass_jet_jet
3,DER_prodeta_jet_jet
4,PRI_jet_subleading_eta
5,PRI_jet_subleading_phi
6,PRI_jet_subleading_pt


## Imputacion por mediana en las columnas restantes

Despues de eliminar las features con mas de `70%` de `-999`, el resto de las columnas todavia puede contener ese marcador especial. En este paso, esos valores se reemplazan por `NaN` y se imputan con la mediana calculada solo sobre `X_train_i1`, usando `SimpleImputer(strategy='median')`. Luego se replica exactamente la transformacion sobre `X_test_i1`.

In [6]:
X_train_i1_for_imputer = X_train_i1.replace(missing_marker, np.nan)
X_test_i1_for_imputer = X_test_i1.replace(missing_marker, np.nan)

num_imputer = SimpleImputer(strategy='median')

X_train_i1_imputed = pd.DataFrame(
    num_imputer.fit_transform(X_train_i1_for_imputer),
    columns=feature_columns_i1,
    index=X_train_i1.index,
)

X_test_i1_imputed = pd.DataFrame(
    num_imputer.transform(X_test_i1_for_imputer),
    columns=feature_columns_i1,
    index=X_test_i1.index,
)

remaining_marker_count_train = int((X_train_i1 == missing_marker).sum().sum())
remaining_marker_count_test = int((X_test_i1 == missing_marker).sum().sum())
total_cells_train_i1 = int(X_train_i1.shape[0] * X_train_i1.shape[1])
total_cells_test_i1 = int(X_test_i1.shape[0] * X_test_i1.shape[1])
remaining_marker_pct_train = remaining_marker_count_train / total_cells_train_i1 * 100
remaining_marker_pct_test = remaining_marker_count_test / total_cells_test_i1 * 100
remaining_nan_count_train = int(X_train_i1_imputed.isna().sum().sum())
remaining_nan_count_test = int(X_test_i1_imputed.isna().sum().sum())

print(
    f"Marcadores -999 restantes en X_train_i1 antes de imputar: {remaining_marker_count_train} "
    f"({remaining_marker_pct_train:.2f}% del total de celdas)"
)
print(
    f"Marcadores -999 restantes en X_test_i1 antes de imputar: {remaining_marker_count_test} "
    f"({remaining_marker_pct_test:.2f}% del total de celdas)"
)
print(f"NaN restantes en X_train_i1_imputed: {remaining_nan_count_train}")
print(f"NaN restantes en X_test_i1_imputed: {remaining_nan_count_test}")


Marcadores -999 restantes en X_train_i1 antes de imputar: 337853 (5.88% del total de celdas)
Marcadores -999 restantes en X_test_i1 antes de imputar: 744290 (5.88% del total de celdas)
NaN restantes en X_train_i1_imputed: 0
NaN restantes en X_test_i1_imputed: 0


## One-hot encoding de `PRI_jet_num`

La columna `PRI_jet_num` representa una cantidad discreta de jets. En lugar de tratarla como una variable numerica continua, en este paso se transforma con `OneHotEncoder` para generar una columna binaria por categoria observada en training. La codificacion se ajusta solo con training y despues se replica sobre test.

In [7]:
jet_num_column = "PRI_jet_num"
remaining_numeric_columns_i1 = [column for column in feature_columns_i1 if column != jet_num_column]

jet_num_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')

X_train_i1_jet_num = X_train_i1_imputed[[jet_num_column]].astype(int)
X_test_i1_jet_num = X_test_i1_imputed[[jet_num_column]].astype(int)

X_train_i1_jet_num_encoded_array = jet_num_encoder.fit_transform(X_train_i1_jet_num)
jet_num_encoded_columns = jet_num_encoder.get_feature_names_out([jet_num_column]).tolist()

X_train_i1_jet_num_encoded = pd.DataFrame(
    X_train_i1_jet_num_encoded_array,
    columns=jet_num_encoded_columns,
    index=X_train_i1_imputed.index,
)

X_test_i1_jet_num_encoded = pd.DataFrame(
    jet_num_encoder.transform(X_test_i1_jet_num),
    columns=jet_num_encoded_columns,
    index=X_test_i1_imputed.index,
)

print(f"Columnas generadas para {jet_num_column}: {jet_num_encoded_columns}")
print(f"X_train_i1_jet_num_encoded shape = {X_train_i1_jet_num_encoded.shape}")
print(f"X_test_i1_jet_num_encoded shape = {X_test_i1_jet_num_encoded.shape}")


Columnas generadas para PRI_jet_num: ['PRI_jet_num_0', 'PRI_jet_num_1', 'PRI_jet_num_2', 'PRI_jet_num_3']
X_train_i1_jet_num_encoded shape = (250000, 4)
X_test_i1_jet_num_encoded shape = (550000, 4)


## Estandarizacion de las columnas

Despues de imputar y codificar `PRI_jet_num`, se aplica `StandardScaler` sobre las columnas numericas restantes. Las columnas binarias generadas por el one-hot encoding se conservan en `0/1` sin escalar. El escalador se ajusta solo con training y despues se usa para transformar tanto training como test.

In [8]:
num_scaler = StandardScaler()

X_train_i1_scaled_numeric = pd.DataFrame(
    num_scaler.fit_transform(X_train_i1_imputed[remaining_numeric_columns_i1]),
    columns=remaining_numeric_columns_i1,
    index=X_train_i1_imputed.index,
)

X_test_i1_scaled_numeric = pd.DataFrame(
    num_scaler.transform(X_test_i1_imputed[remaining_numeric_columns_i1]),
    columns=remaining_numeric_columns_i1,
    index=X_test_i1_imputed.index,
)

X_train_i1_scaled = pd.concat([X_train_i1_scaled_numeric, X_train_i1_jet_num_encoded], axis=1)
X_test_i1_scaled = pd.concat([X_test_i1_scaled_numeric, X_test_i1_jet_num_encoded], axis=1)
feature_columns_i1_final = X_train_i1_scaled.columns.tolist()

scaler_summary = pd.DataFrame(
    {
        "train_mean_after_scaling": X_train_i1_scaled_numeric.mean().round(4),
        "train_std_after_scaling": X_train_i1_scaled_numeric.std(ddof=0).round(4),
    }
)

print(f"X_train_i1_scaled shape = {X_train_i1_scaled.shape}")
print(f"X_test_i1_scaled shape = {X_test_i1_scaled.shape}")
print(f"Cantidad final de features despues de one-hot encoding: {len(feature_columns_i1_final)}")
display(scaler_summary.head(10))


X_train_i1_scaled shape = (250000, 26)
X_test_i1_scaled shape = (550000, 26)
Cantidad final de features despues de one-hot encoding: 26


,train_mean_after_scaling,train_std_after_scaling
DER_mass_MMC,0.0,1.0
DER_mass_transverse_met_lep,0.0,1.0
DER_mass_vis,-0.0,1.0
DER_pt_h,0.0,1.0
DER_deltar_tau_lep,-0.0,1.0
DER_pt_tot,-0.0,1.0
DER_sum_pt,-0.0,1.0
DER_pt_ratio_lep_tau,0.0,1.0
DER_met_phi_centrality,-0.0,1.0
PRI_tau_pt,0.0,1.0


## Objetos resultantes del intento 1

Se muestran las primeras filas de `X_train_i1_scaled` y de `y_train` para confirmar que la separacion sigue bien y que el intento 1 queda listo con descarte de columnas muy incompletas, imputacion por mediana, one-hot encoding de `PRI_jet_num` y estandarizacion del resto de las features.

In [9]:
display(X_train_i1_scaled.head())
display(y_train.head().to_frame(name="Label"))


,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltar_tau_lep,DER_pt_tot,DER_sum_pt,DER_pt_ratio_lep_tau,DER_met_phi_centrality,PRI_tau_pt,...,PRI_met_phi,PRI_met_sumet,PRI_jet_leading_pt,PRI_jet_leading_eta,PRI_jet_leading_phi,PRI_jet_all_pt,PRI_jet_num_0,PRI_jet_num_1,PRI_jet_num_2,PRI_jet_num_3
0,0.341522,0.068332,0.407680,-0.469966,0.882478,1.033099,0.339894,0.170929,1.277084,-0.270811,...,-0.147267,0.386847,-0.202115,1.556350,0.330677,0.412510,0.0,0.0,1.0,0.0
1,0.766558,0.552505,0.540136,-0.153167,1.404888,-0.756027,-0.287584,-0.661279,1.292164,0.147536,...,-1.051683,-0.357719,-0.644524,0.525758,0.838833,-0.273820,0.0,1.0,0.0,0.0
2,-0.151562,3.195156,1.096560,-0.349710,0.989770,-0.430168,0.340361,2.768174,1.292164,-0.292406,...,-1.200672,0.400135,-0.685722,1.486197,-1.428652,-0.293970,0.0,1.0,0.0,0.0
3,0.444343,0.910379,-0.005853,-0.903016,1.196690,-0.830735,-0.712705,1.084818,-0.969095,-0.716598,...,0.038692,-0.978149,-0.241206,0.001422,-0.008805,-0.745439,1.0,0.0,0.0,0.0
4,1.048950,-0.914556,1.313369,-0.651804,1.938794,-0.112795,-0.868143,-0.451747,-1.052877,-0.468428,...,-0.475042,-1.238475,-0.241206,0.001422,-0.008805,-0.745439,1.0,0.0,0.0,0.0


,Label
0,s
1,b
2,b
3,b
4,b


## Pipeline final reutilizable

A continuacion se deja una version reusable del flujo completo del intento 1. Esta version parte desde la carga de datos, separa columnas especiales, detecta las features a descartar por exceso de `-999`, construye un `Pipeline` de scikit-learn con `ColumnTransformer` y devuelve los datasets transformados listos para entrenar modelos.

In [10]:
def load_higgs_data(project_root: Path | None = None) -> tuple[pd.DataFrame, pd.DataFrame]:
    root = find_project_root(project_root)
    data_dir = root / "data" / "higgs-boson"
    train_path = data_dir / "training.csv"
    test_path = data_dir / "test.csv"

    if not train_path.exists():
        raise FileNotFoundError(f"No se encontro el archivo esperado: {train_path}")

    if not test_path.exists():
        raise FileNotFoundError(f"No se encontro el archivo esperado: {test_path}")

    return pd.read_csv(train_path), pd.read_csv(test_path)


def split_higgs_datasets(
    train_data: pd.DataFrame,
    test_data: pd.DataFrame,
) -> tuple[pd.Series, pd.Series, pd.Series, pd.Series, pd.DataFrame, pd.DataFrame]:
    excluded_train_columns = ["EventId", "Weight", "Label"]
    feature_columns = [
        column for column in train_data.columns
        if column not in excluded_train_columns
    ]

    expected_test_columns = [column for column in test_data.columns if column != "EventId"]
    if feature_columns != expected_test_columns:
        raise ValueError("Las columnas de entrenamiento y test no coinciden despues de excluir EventId.")

    train_event_id = train_data["EventId"].copy()
    train_weight = train_data["Weight"].copy()
    y_train = train_data["Label"].copy()
    X_train = train_data[feature_columns].copy()

    test_event_id = test_data["EventId"].copy()
    X_test = test_data[feature_columns].copy()

    return train_event_id, train_weight, y_train, test_event_id, X_train, X_test


def select_attempt1_feature_columns(
    X_train: pd.DataFrame,
    missing_marker: float = -999,
    missing_threshold_pct: float = 70.0,
) -> tuple[list[str], list[str], pd.DataFrame]:
    marker_summary = pd.DataFrame(
        {
            "feature": X_train.columns,
            "train_minus_999_count": [(X_train[column] == missing_marker).sum() for column in X_train.columns],
        }
    )
    marker_summary["train_minus_999_pct"] = (
        marker_summary["train_minus_999_count"] / len(X_train) * 100
    ).round(2)
    marker_summary = marker_summary.sort_values(
        ["train_minus_999_pct", "feature"],
        ascending=[False, True],
    )

    drop_columns = marker_summary.loc[
        marker_summary["train_minus_999_pct"] > missing_threshold_pct,
        "feature",
    ].tolist()
    kept_columns = [column for column in X_train.columns if column not in drop_columns]

    return kept_columns, drop_columns, marker_summary


def build_attempt1_preprocessing_pipeline(
    feature_columns: list[str],
    jet_num_column: str = "PRI_jet_num",
    missing_marker: float = -999,
) -> tuple[Pipeline, list[str], list[str]]:
    if jet_num_column not in feature_columns:
        raise ValueError(f"La columna {jet_num_column} debe existir en feature_columns.")

    numeric_columns = [column for column in feature_columns if column != jet_num_column]

    numeric_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(missing_values=missing_marker, strategy="median")),
            ("scaler", StandardScaler()),
        ]
    )

    jet_num_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("one_hot", OneHotEncoder(sparse_output=False, handle_unknown="ignore")),
        ]
    )

    preprocessor = ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_columns),
            ("jet_num", jet_num_pipeline, [jet_num_column]),
        ],
        remainder="drop",
        verbose_feature_names_out=False,
    )

    preprocessing_pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
        ]
    )

    return preprocessing_pipeline, numeric_columns, [jet_num_column]


def transform_attempt1_with_pipeline(
    train_data: pd.DataFrame,
    test_data: pd.DataFrame,
    missing_marker: float = -999,
    missing_threshold_pct: float = 70.0,
) -> dict[str, object]:
    train_event_id, train_weight, y_train, test_event_id, X_train, X_test = split_higgs_datasets(
        train_data,
        test_data,
    )

    feature_columns_i1, drop_columns_i1, marker_summary_i1 = select_attempt1_feature_columns(
        X_train,
        missing_marker=missing_marker,
        missing_threshold_pct=missing_threshold_pct,
    )

    preprocessing_pipeline, numeric_columns_i1, categorical_columns_i1 = build_attempt1_preprocessing_pipeline(
        feature_columns_i1,
        missing_marker=missing_marker,
    )

    X_train_selected = X_train[feature_columns_i1].copy()
    X_test_selected = X_test[feature_columns_i1].copy()

    X_train_processed_array = preprocessing_pipeline.fit_transform(X_train_selected)
    X_test_processed_array = preprocessing_pipeline.transform(X_test_selected)

    processed_feature_names = preprocessing_pipeline.named_steps["preprocessor"].get_feature_names_out().tolist()

    X_train_processed = pd.DataFrame(
        X_train_processed_array,
        columns=processed_feature_names,
        index=X_train_selected.index,
    )

    X_test_processed = pd.DataFrame(
        X_test_processed_array,
        columns=processed_feature_names,
        index=X_test_selected.index,
    )

    return {
        "train_event_id": train_event_id,
        "train_weight": train_weight,
        "y_train": y_train,
        "test_event_id": test_event_id,
        "feature_columns_i1": feature_columns_i1,
        "drop_columns_i1": drop_columns_i1,
        "marker_summary_i1": marker_summary_i1,
        "numeric_columns_i1": numeric_columns_i1,
        "categorical_columns_i1": categorical_columns_i1,
        "processed_feature_names": processed_feature_names,
        "preprocessing_pipeline": preprocessing_pipeline,
        "X_train_processed": X_train_processed,
        "X_test_processed": X_test_processed,
    }


train_data_pipeline, test_data_pipeline = load_higgs_data()
attempt1_pipeline_artifacts = transform_attempt1_with_pipeline(train_data_pipeline, test_data_pipeline)

print(f"Columnas descartadas en intento 1: {attempt1_pipeline_artifacts['drop_columns_i1']}")
print(f"Cantidad de features conservadas antes del pipeline: {len(attempt1_pipeline_artifacts['feature_columns_i1'])}")
print(f"Cantidad final de features procesadas: {len(attempt1_pipeline_artifacts['processed_feature_names'])}")
print(f"X_train_processed shape = {attempt1_pipeline_artifacts['X_train_processed'].shape}")
print(f"X_test_processed shape = {attempt1_pipeline_artifacts['X_test_processed'].shape}")

display(attempt1_pipeline_artifacts['X_train_processed'].head())
display(attempt1_pipeline_artifacts['y_train'].head().to_frame(name='Label'))


Columnas descartadas en intento 1: ['DER_deltaeta_jet_jet', 'DER_lep_eta_centrality', 'DER_mass_jet_jet', 'DER_prodeta_jet_jet', 'PRI_jet_subleading_eta', 'PRI_jet_subleading_phi', 'PRI_jet_subleading_pt']
Cantidad de features conservadas antes del pipeline: 23
Cantidad final de features procesadas: 26
X_train_processed shape = (250000, 26)
X_test_processed shape = (550000, 26)


,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltar_tau_lep,DER_pt_tot,DER_sum_pt,DER_pt_ratio_lep_tau,DER_met_phi_centrality,PRI_tau_pt,...,PRI_met_phi,PRI_met_sumet,PRI_jet_leading_pt,PRI_jet_leading_eta,PRI_jet_leading_phi,PRI_jet_all_pt,PRI_jet_num_0,PRI_jet_num_1,PRI_jet_num_2,PRI_jet_num_3
0,0.341522,0.068332,0.407680,-0.469966,0.882478,1.033099,0.339894,0.170929,1.277084,-0.270811,...,-0.147267,0.386847,-0.202115,1.556350,0.330677,0.412510,0.0,0.0,1.0,0.0
1,0.766558,0.552505,0.540136,-0.153167,1.404888,-0.756027,-0.287584,-0.661279,1.292164,0.147536,...,-1.051683,-0.357719,-0.644524,0.525758,0.838833,-0.273820,0.0,1.0,0.0,0.0
2,-0.151562,3.195156,1.096560,-0.349710,0.989770,-0.430168,0.340361,2.768174,1.292164,-0.292406,...,-1.200672,0.400135,-0.685722,1.486197,-1.428652,-0.293970,0.0,1.0,0.0,0.0
3,0.444343,0.910379,-0.005853,-0.903016,1.196690,-0.830735,-0.712705,1.084818,-0.969095,-0.716598,...,0.038692,-0.978149,-0.241206,0.001422,-0.008805,-0.745439,1.0,0.0,0.0,0.0
4,1.048950,-0.914556,1.313369,-0.651804,1.938794,-0.112795,-0.868143,-0.451747,-1.052877,-0.468428,...,-0.475042,-1.238475,-0.241206,0.001422,-0.008805,-0.745439,1.0,0.0,0.0,0.0


,Label
0,s
1,b
2,b
3,b
4,b


## Guardado de los datos procesados

Para cerrar el intento 1, se guardan los datasets procesados y los artefactos principales en `data/higgs-boson/processed/i1/`. Esto deja disponible una salida reutilizable para los siguientes pasos de modelado sin tener que reejecutar manualmente todas las transformaciones.

In [11]:
def save_attempt1_processed_data(
    artifacts: dict[str, object],
    output_dir: Path,
) -> list[Path]:
    output_dir.mkdir(parents=True, exist_ok=True)

    files_to_save: dict[str, pd.DataFrame] = {
        "X_train_processed.csv": artifacts["X_train_processed"],
        "X_test_processed.csv": artifacts["X_test_processed"],
        "y_train.csv": artifacts["y_train"].to_frame(name="Label"),
        "train_weight.csv": artifacts["train_weight"].to_frame(name="Weight"),
        "train_event_id.csv": artifacts["train_event_id"].to_frame(name="EventId"),
        "test_event_id.csv": artifacts["test_event_id"].to_frame(name="EventId"),
        "drop_columns_i1.csv": pd.DataFrame({"dropped_feature": artifacts["drop_columns_i1"]}),
        "feature_columns_i1.csv": pd.DataFrame({"feature": artifacts["feature_columns_i1"]}),
        "processed_feature_names.csv": pd.DataFrame({"feature": artifacts["processed_feature_names"]}),
        "marker_summary_i1.csv": artifacts["marker_summary_i1"],
    }

    saved_paths: list[Path] = []
    for filename, dataframe in files_to_save.items():
        file_path = output_dir / filename
        dataframe.to_csv(file_path, index=False)
        saved_paths.append(file_path)

    return saved_paths


processed_output_dir = find_project_root() / "data" / "higgs-boson" / "processed" / "i1"
saved_paths_i1 = save_attempt1_processed_data(attempt1_pipeline_artifacts, processed_output_dir)

print(f"Directorio de salida: {processed_output_dir}")
print(f"Cantidad de archivos guardados: {len(saved_paths_i1)}")
for path in saved_paths_i1:
    print(f"- {path}")


Directorio de salida: /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i1
Cantidad de archivos guardados: 10
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i1/X_train_processed.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i1/X_test_processed.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i1/y_train.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i1/train_weight.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i1/train_event_id.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i1/test_event_id.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i1/drop_columns_i1.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i1/feature_columns_i1.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i1/processed_feature_names.csv
- /home/santig14/fing/taa/1_TallerAA/data/higgs-boson/processed/i1/marker_summary_i1.csv
